# 01 - Data Cleaning
### Bank Customer Segmentation using PCA + K-Means

This notebook loads the **raw, uncleaned** transaction data (`data/raw/bank_transactions_raw.csv`)
and produces a clean dataset saved to `data/processed/bank_transactions_clean.csv`.

Issues handled:
1. Missing values (DOB, Gender, Location, Balance, Amount)
2. Inconsistent Gender labels / casing
3. Currency strings with symbols & commas → numeric
4. Inconsistent date formats across rows
5. Unrealistic / corrupted DOBs
6. Whitespace & casing issues in city names
7. Exact duplicate rows
8. Outliers / erroneous negative or extreme transaction amounts


In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

raw = pd.read_csv('../data/raw/bank_transactions_raw.csv')
print(raw.shape)
raw.head()

(6180, 9)


,TransactionID,CustomerID,CustomerDOB,CustGender,CustLocation,CustAccountBalance,TransactionDate,TransactionTime,TransactionAmount (INR)
0,T502535,C100726,1966-04-07,m,KOLKATA,"Rs. 213,622.83",2024-06-12,230909,6148.9
1,T500740,C100796,13-01-85,F,CHENNAI,22196.84,01-Jun-2024,42021,1284.31
2,T501682,C100135,1976-12-12,Female,mumbai,NaN,11/05/2024,183515,323.89
3,T501409,C100163,04/08/1970,m,Kolkata,"Rs. 3,126.03",11-Jul-2024,125144,320.57
4,T505100,C100509,1977-09-18,m,BANGALORE,26256.74,20-Mar-2024,31024,789.35


## 1. Initial inspection

In [2]:
raw.info()
print('\nMissing values per column:')
print(raw.isna().sum())
print('\nExact duplicate rows:', raw.duplicated().sum())

<class 'pandas.DataFrame'>
RangeIndex: 6180 entries, 0 to 6179
Data columns (total 9 columns):
 #   Column                   Non-Null Count  Dtype
---  ------                   --------------  -----
 0   TransactionID            6180 non-null   str  
 1   CustomerID               6180 non-null   str  
 2   CustomerDOB              6138 non-null   str  
 3   CustGender               5476 non-null   str  
 4   CustLocation             5889 non-null   str  
 5   CustAccountBalance       5167 non-null   str  
 6   TransactionDate          6180 non-null   str  
 7   TransactionTime          6180 non-null   int64
 8   TransactionAmount (INR)  5991 non-null   str  
dtypes: int64(1), str(8)
memory usage: 788.5 KB

Missing values per column:
TransactionID                 0
CustomerID                    0
CustomerDOB                  42
CustGender                  704
CustLocation                291
CustAccountBalance         1013
TransactionDate               0
TransactionTime               0
T

## 2. Drop exact duplicate rows

In [3]:
df = raw.drop_duplicates().copy()
print('Shape after dropping duplicates:', df.shape)

Shape after dropping duplicates: (6000, 9)


## 3. Clean `CustGender`

In [4]:
gender_map = {
    'm': 'Male', 'male': 'Male', 'M': 'Male', 'Male': 'Male',
    'f': 'Female', 'female': 'Female', 'F': 'Female', 'Female': 'Female'
}
df['CustGender'] = df['CustGender'].astype(str).str.strip().str.lower().map(
    {'m':'Male','male':'Male','f':'Female','female':'Female'}
)
df['CustGender'] = df['CustGender'].fillna('Unknown')
df['CustGender'].value_counts(dropna=False)

CustGender
Male       2692
Female     2628
Unknown     680
Name: count, dtype: int64

## 4. Clean `CustLocation`

In [5]:
df['CustLocation'] = df['CustLocation'].astype(str).str.strip().str.title()
df.loc[df['CustLocation'].isin(['Nan', 'None', '']), 'CustLocation'] = np.nan
df['CustLocation'] = df['CustLocation'].fillna('Unknown')
df['CustLocation'].value_counts().head(10)

CustLocation
Mumbai       722
Pune         630
Jaipur       539
Delhi        535
Chennai      512
Ahmedabad    485
Kolkata      483
Hyderabad    449
Bangalore    384
Lucknow      311
Name: count, dtype: int64

## 5. Clean `CustAccountBalance` (strip currency symbols/commas → float)

In [6]:
def clean_currency(x):
    if pd.isna(x):
        return np.nan
    if isinstance(x, (int, float)):
        return float(x)
    x = str(x).replace('Rs.', '').replace('INR', '').replace(',', '').strip()
    try:
        return float(x)
    except ValueError:
        return np.nan

df['CustAccountBalance'] = df['CustAccountBalance'].apply(clean_currency)
df['TransactionAmount (INR)'] = df['TransactionAmount (INR)'].apply(clean_currency)

print(df[['CustAccountBalance', 'TransactionAmount (INR)']].describe())

       CustAccountBalance  TransactionAmount (INR)
count         5017.000000             5.818000e+03
mean         84456.434046             6.687253e+03
std         107927.861337             4.333437e+04
min            296.870000            -2.141820e+05
25%          21365.930000             8.023575e+02
50%          50615.700000             2.001270e+03
75%         101026.530000             4.541455e+03
max         772078.460000             1.272585e+06


## 6. Fix erroneous transaction amounts (negative / extreme outliers)

In [7]:
# Negative amounts are data-entry errors here -> take absolute value
df['TransactionAmount (INR)'] = df['TransactionAmount (INR)'].abs()

# Cap extreme outliers at the 99.5th percentile (winsorize) instead of dropping rows
upper_cap = df['TransactionAmount (INR)'].quantile(0.995)
df['TransactionAmount (INR)'] = np.where(
    df['TransactionAmount (INR)'] > upper_cap, upper_cap, df['TransactionAmount (INR)']
)
print('Upper cap used:', upper_cap)

Upper cap used: 272883.0199999997


## 7. Parse inconsistent dates

In [8]:
def parse_date(x):
    return pd.to_datetime(x, errors='coerce', dayfirst=True)

df['TransactionDate'] = df['TransactionDate'].apply(parse_date)
df['CustomerDOB'] = df['CustomerDOB'].apply(parse_date)

# Unrealistic DOBs (before 1930 or after 2010) -> treat as missing
mask_bad_dob = (df['CustomerDOB'].dt.year < 1930) | (df['CustomerDOB'].dt.year > 2010)
df.loc[mask_bad_dob, 'CustomerDOB'] = pd.NaT

print('Missing TransactionDate after parsing:', df['TransactionDate'].isna().sum())
print('Missing CustomerDOB after parsing:', df['CustomerDOB'].isna().sum())

/tmp/ipykernel_616/268159784.py:2: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  return pd.to_datetime(x, errors='coerce', dayfirst=True)


/tmp/ipykernel_616/268159784.py:2: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  return pd.to_datetime(x, errors='coerce', dayfirst=True)


Missing TransactionDate after parsing: 0
Missing CustomerDOB after parsing: 984


## 8. Derive `Age` from DOB, impute remaining missing values

In [9]:
reference_date = pd.Timestamp('2024-06-30')
df['Age'] = ((reference_date - df['CustomerDOB']).dt.days / 365.25)

# impute missing Age with median age
df['Age'] = df['Age'].fillna(df['Age'].median()).round(1)
df = df[(df['Age'] >= 15) & (df['Age'] <= 90)]  # sanity bounds

# impute missing balance with median balance per gender group
df['CustAccountBalance'] = df.groupby('CustGender')['CustAccountBalance']\
    .transform(lambda s: s.fillna(s.median()))

# impute any remaining missing amount with column median
df['TransactionAmount (INR)'] = df['TransactionAmount (INR)'].fillna(
    df['TransactionAmount (INR)'].median()
)

# drop rows still missing an essential transaction date
df = df.dropna(subset=['TransactionDate'])

print(df.isna().sum())
print('Final shape:', df.shape)

TransactionID                0
CustomerID                   0
CustomerDOB                984
CustGender                   0
CustLocation                 0
CustAccountBalance           0
TransactionDate              0
TransactionTime              0
TransactionAmount (INR)      0
Age                          0
dtype: int64
Final shape: (6000, 10)


## 9. Final checks & save cleaned dataset

In [10]:
df = df.reset_index(drop=True)
df.to_csv('../data/processed/bank_transactions_clean.csv', index=False)
df.head()

,TransactionID,CustomerID,CustomerDOB,CustGender,CustLocation,CustAccountBalance,TransactionDate,TransactionTime,TransactionAmount (INR),Age
0,T502535,C100726,1966-07-04,Male,Kolkata,213622.83,2024-12-06,230909,6148.90,58.0
1,T500740,C100796,1985-01-13,Female,Chennai,22196.84,2024-06-01,42021,1284.31,39.5
2,T501682,C100135,1976-12-12,Female,Mumbai,52253.95,2024-05-11,183515,323.89,47.5
3,T501409,C100163,1970-08-04,Male,Kolkata,3126.03,2024-07-11,125144,320.57,53.9
4,T505100,C100509,1977-09-18,Male,Bangalore,26256.74,2024-03-20,31024,789.35,46.8
